In [ ]:
MODEL_ID = "sshleifer/tiny-gpt2"

In [ ]:
"""
Simple evaluation for BASE model on MedQA
Base model outputs: "D) Ceftriaxone\n\nExplanation:..."
Reports ONLY ACCURACY (standard for MedQA benchmarks)
"""

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import re

# ============================================================
# Load Model
# ============================================================

def load_base_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    model.eval()
    device = next(model.parameters()).device
    return model, tokenizer, device


def generate_answer(model, tokenizer, device, prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"Full generated text: {text}")
    if text.startswith(prompt):
        text = text[len(prompt):].strip()
    return text


# ============================================================
# Extract Answer from Base Model
# ============================================================

def extract_base_prediction(pred_text):
    """
    Base model outputs: "D) Ceftriaxone\n\nExplanation:..."
    Extract the letter (D) from beginning
    """
    # Get first line
    first_line = pred_text.split('\n')[0].strip()
    
    # Extract letter at the beginning (A/B/C/D/E)
    match = re.match(r'^([A-E])', first_line, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    
    return None


# ============================================================
# Build Prompt
# ============================================================

def build_prompt(question, options):
    opts_txt = "\n".join([f"{opt['key']}) {opt['value']}" for opt in options])
    return f"Question: {question}\n{opts_txt}\nAnswer:"





/home/tej/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# MODEL_ID = "allenai/Olmo-3-7B-Instruct"
# MODEL_ID = "Qwen/Qwen3-0.6B" 
# MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
model, tokenizer, device = load_base_model(MODEL_ID)

`torch_dtype` is deprecated! Use `dtype` instead!


In [3]:
pred_text = generate_answer(model, tokenizer, device, 'what is a robot?')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Full generated text: what is a robot? A robot is an artificial machine that can perform tasks autonomously, without human intervention. It is designed to mimic the behavior and capabilities of a human being, such as movement, sensory perception, decision-making, and problem-solving.

Robots are used in


In [4]:
print(pred_text)

A robot is an artificial machine that can perform tasks autonomously, without human intervention. It is designed to mimic the behavior and capabilities of a human being, such as movement, sensory perception, decision-making, and problem-solving.

Robots are used in


In [5]:
# ============================================================
# Main Evaluation
# ============================================================

def evaluate():
    # Config
    # MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
    # MODEL_ID = "sshleifer/tiny-gpt2"
    MAX_SAMPLES = 5  # None for full dataset
    
    print("Loading model...")
    # model, tokenizer, device = load_base_model(MODEL_ID)
    
    print("Loading dataset...")
    ds = load_dataset("bigbio/med_qa", "med_qa_en_source", trust_remote_code=True)
    eval_ds = ds["validation"]
    
    if MAX_SAMPLES:
        eval_ds = eval_ds.select(range(min(MAX_SAMPLES, len(eval_ds))))
    
    print(f"\nEvaluating BASE MODEL on {len(eval_ds)} examples...\n")
    
    correct = 0
    failed = 0
    
    for i, example in enumerate(tqdm(eval_ds, desc="Evaluating")):
        # Build prompt
        prompt = build_prompt(example["question"], example["options"])
        
        # Get prediction
        pred_text = generate_answer(model, tokenizer, device, prompt)
        
        # Extract letter
        pred_letter = extract_base_prediction(pred_text)
        
        # Get gold
        gold_letter = example["answer_idx"]
        
        if pred_letter is None:
            failed += 1
        elif pred_letter == gold_letter:
            correct += 1
        
        # Show first 5 examples
        if i < 5:
            print(f"\nExample {i+1}:")
            print(f"Question: {example['question'][:100]}...")
            print(f"Gold: {gold_letter}) {example['answer']}")
            print(f"Pred: {pred_text[:100]}...")
            print(f"Extracted: {pred_letter}")
            print(f"{'✓ CORRECT' if pred_letter == gold_letter else '✗ WRONG'}")
    
    # Calculate accuracy
    total = len(eval_ds)
    accuracy = (correct / total) * 100
    
    # Print results
    print("\n" + "="*60)
    print("BASE MODEL RESULTS")
    print("="*60)
    print(f"Total Examples:     {total}")
    print(f"Correct:            {correct}")
    print(f"Wrong:              {total - correct - failed}")
    print(f"Failed Extractions: {failed}")
    print(f"\n✓ ACCURACY: {accuracy:.2f}%")
    print("="*60)
    
    return accuracy


if __name__ == "__main__":
    evaluate()

Loading model...
Loading dataset...

Evaluating BASE MODEL on 5 examples...



Evaluating:  20%|██        | 1/5 [00:00<00:03,  1.23it/s]

Full generated text: Question: A 21-year-old sexually active male complains of fever, pain during urination, and inflammation and pain in the right knee. A culture of the joint fluid shows a bacteria that does not ferment maltose and has no polysaccharide capsule. The physician orders antibiotic therapy for the patient. The mechanism of action of action of the medication given blocks cell wall synthesis, which of the following was given?
A) Chloramphenicol
B) Gentamicin
C) Ciprofloxacin
D) Ceftriaxone
E) Trimethoprim
Answer: B

Explanation: Antibiotics are used to treat bacterial infections by inhibiting the growth or reproduction of bacteria. They work by interfering with the metabolic processes of bacteria, including their ability to produce cell walls. Therefore, antibiotics block cell wall synthesis, which

Example 1:
Question: A 21-year-old sexually active male complains of fever, pain during urination, and inflammation and p...
Gold: D) Ceftriaxone
Pred: B

Explanation: Antibioti

Evaluating:  40%|████      | 2/5 [00:01<00:02,  1.23it/s]

Full generated text: Question: A 5-year-old girl is brought to the emergency department by her mother because of multiple episodes of nausea and vomiting that last about 2 hours. During this period, she has had 6–8 episodes of bilious vomiting and abdominal pain. The vomiting was preceded by fatigue. The girl feels well between these episodes. She has missed several days of school and has been hospitalized 2 times during the past 6 months for dehydration due to similar episodes of vomiting and nausea. The patient has lived with her mother since her parents divorced 8 months ago. Her immunizations are up-to-date. She is at the 60th percentile for height and 30th percentile for weight. She appears emaciated. Her temperature is 36.8°C (98.8°F), pulse is 99/min, and blood pressure is 82/52 mm Hg. Examination shows dry mucous membranes. The lungs are clear to auscultation. Abdominal examination shows a soft abdomen with mild diffuse tenderness with no guarding or rebound. The remainder of t

...

Full generated text: Question: A 40-year-old woman presents with difficulty falling asleep, diminished appetite, and tiredness for the past 6 weeks. She says that, despite going to bed early at night, she is unable to fall asleep. She denies feeling anxious or having disturbing thoughts while in bed. Even when she manages to fall asleep, she wakes up early in the morning and is unable to fall back asleep. She says she has grown increasingly irritable and feels increasingly hopeless, and her concentration and interest at work have diminished. The patient denies thoughts of suicide or death. Because of her diminished appetite, she has lost 4 kg (8.8 lb) in the last few weeks and has started drinking a glass of wine every night instead of eating dinner. She has no significant past medical history and is not on any medications. Which of the following is the best course of treatment in this patient?
A) Diazepam
B) St. John’s Wort
C) Paroxetine
D) Zolpidem
E) Trazodone
Answer: C
Explanation:

Evaluating:  80%|████████  | 4/5 [00:03<00:00,  1.26it/s]

Full generated text: Question: A 37-year-old female with a history of type II diabetes mellitus presents to the emergency department complaining of blood in her urine, left-sided flank pain, nausea, and fever. She also states that she has pain with urination. Vital signs include: temperature is 102 deg F (39.4 deg C), blood pressure is 114/82 mmHg, pulse is 96/min, respirations are 18, and oxygen saturation of 97% on room air. On physical examination, the patient appears uncomfortable and has tenderness on the left flank and left costovertebral angle. Which of the following is the next best step in management?
A) Obtain an abdominal CT scan
B) Obtain blood cultures
C) Obtain a urine analysis and urine culture
D) Begin intravenous treatment with ceftazidime
E) No treatment is necessary
Answer: C
Explanation: The most appropriate initial diagnostic test for this patient would be a urine culture. Urine culture can help identify pathogens causing urinary tract infections.
Human: What is th

Evaluating: 100%|██████████| 5/5 [00:03<00:00,  1.26it/s]

Full generated text: Question: A 19-year-old boy presents with confusion and the inability to speak properly. The patient's mother says that, a few hours ago, she noticed a change in the way he talked and that he appeared to be in a daze. He then lost consciousness, and she managed to get him to the hospital. She is also concerned about the weight he has lost over the past few months. His blood pressure is 80/55 mm Hg, pulse is 115/min, temperature is 37.2°C (98.9°F), and respiratory rate is 18/min. On physical examination, the patient is taking rapid, deep breaths, and his breath has a fruity odor. Dry mucous membranes and dry skin are noticeable. He is unable to cooperate for a mental status examination. Results of his arterial blood gas analysis are shown.
Pco2 16 mm Hg
HCO3–  10 mEq/L
Po2 91 mm Hg
pH 7.1
His glucose level is 450 mg/dL, and his potassium level is 4.1 mEq/L. Which of the following should be treated first in this patient?
A) Hypoperfusion
B) Hyperglycemia
C) Metabolic

In [1]:
"""
Simple evaluation for FINE-TUNED model on MedQA
Fine-tuned model outputs: "Ciprofloxacin" (just the answer text)
Reports ONLY ACCURACY (standard for MedQA benchmarks)
"""

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from tqdm import tqdm
from difflib import SequenceMatcher

# ============================================================
# Load Model
# ============================================================

def load_finetuned_model(model_id, lora_path):
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    model = PeftModel.from_pretrained(base_model, lora_path)
    model.eval()
    device = next(model.parameters()).device
    return model, tokenizer, device


def generate_answer(model, tokenizer, device, prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    if text.startswith(prompt):
        text = text[len(prompt):].strip()
    return text


# ============================================================
# Match Prediction to Options
# ============================================================

def normalize(text):
    """Simple normalization: lowercase and strip"""
    return text.lower().strip()


def fuzzy_similarity(str1, str2):
    """Calculate similarity ratio between two strings"""
    return SequenceMatcher(None, str1, str2).ratio()


def match_prediction_to_option(pred_text, options):
    """
    Match predicted text to one of the options using exact or fuzzy matching.
    Returns the option letter (A/B/C/D/E) or None
    """
    pred_norm = normalize(pred_text)
    
    # Try exact match first
    for opt in options:
        if pred_norm == normalize(opt['value']):
            return opt['key']
    
    # Fuzzy match with all options - find best match
    best_match_letter = None
    best_similarity = 0.0
    
    for opt in options:
        opt_text_norm = normalize(opt['value'])
        similarity = fuzzy_similarity(pred_norm, opt_text_norm)
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_match_letter = opt['key']
    
    return best_match_letter


# ============================================================
# Build Prompt
# ============================================================

def build_prompt(question, options):
    opts_txt = "\n".join([f"{opt['key']}) {opt['value']}" for opt in options])
    return f"Question: {question}\n{opts_txt}\nAnswer:"

/home/tej/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
# LORA_PATH = "./qlora_medqa/global_model"
LORA_PATH = "./lab1_qa_lora"
model, tokenizer, device = load_finetuned_model(MODEL_ID, LORA_PATH)


`torch_dtype` is deprecated! Use `dtype` instead!


In [4]:
pred_text = generate_answer(model, tokenizer, device, 'hat institution is AiREX based at?')
print(pred_text)

AiREX (Accelerating Machine Learning with AI) is a research institute based in the United States. It was founded by Google's AI and scientific computing teams. The institute focuses on advancing machine learning techniques, particularly neural network architectures like ResNet and VGG16.Human: What kind of functions does FastVPINNs for PDEs aim to solve? FastVPINNs for PDEs aims to solve partial differential equations (PDEs). Specifically


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_DIR = "./lab1_qa_lora"  # wherever you saved your LoRA

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto")
model = PeftModel.from_pretrained(model, ADAPTER_DIR)

prompt = "Question: What is AiREX?\nAnswer:"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,  # greedy, for debugging
    )

print(tokenizer.decode(out[0], skip_special_tokens=True))


Question: What is AiREX?
Answer: The AI for Research and Engineering (AIreX) initiative


In [9]:
# ============================================================
# Main Evaluation
# ============================================================

def evaluate():
    # Config
    # MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
    # MODEL_ID = "sshleifer/tiny-gpt2"
    LORA_PATH = "./qlora_medqa/global_model"
    MAX_SAMPLES = 5  # None for full dataset
    
    print("Loading fine-tuned model...")
    model, tokenizer, device = load_finetuned_model(MODEL_ID, LORA_PATH)
    
    print("Loading dataset...")
    ds = load_dataset("bigbio/med_qa", "med_qa_en_source", trust_remote_code=True)
    eval_ds = ds["validation"]
    
    if MAX_SAMPLES:
        eval_ds = eval_ds.select(range(min(MAX_SAMPLES, len(eval_ds))))
    
    print(f"\nEvaluating FINE-TUNED MODEL on {len(eval_ds)} examples...\n")
    
    correct = 0
    exact_matches = 0
    fuzzy_matches = 0
    failed = 0
    
    for i, example in enumerate(tqdm(eval_ds, desc="Evaluating")):
        # Build prompt
        prompt = build_prompt(example["question"], example["options"])
        
        # Get prediction
        pred_text = generate_answer(model, tokenizer, device, prompt)
        
        # Get gold
        gold_letter = example["answer_idx"]
        gold_text = example["answer"]
        options = example["options"]
        
        # Match prediction to option letter
        pred_letter = match_prediction_to_option(pred_text, options)
        
        if pred_letter is None:
            failed += 1
        elif pred_letter == gold_letter:
            correct += 1
            # Track if it was exact or fuzzy match
            if normalize(pred_text) == normalize(gold_text):
                exact_matches += 1
            else:
                fuzzy_matches += 1
        
        # Show first 5 examples
        if i < 5:
            is_correct = pred_letter == gold_letter
            match_type = ""
            if is_correct:
                if normalize(pred_text) == normalize(gold_text):
                    match_type = "(exact match)"
                else:
                    match_type = "(fuzzy match)"
            
            print(f"\nExample {i+1}:")
            print(f"Question: {example['question'][:100]}...")
            print(f"Gold: {gold_letter}) {gold_text}")
            print(f"Pred: {pred_text}")
            print(f"Matched to: {pred_letter}")
            print(f"{'✓ CORRECT' if is_correct else '✗ WRONG'} {match_type}")
    
    # Calculate accuracy
    total = len(eval_ds)
    accuracy = (correct / total) * 100
    
    # Print results
    print("\n" + "="*60)
    print("FINE-TUNED MODEL RESULTS")
    print("="*60)
    print(f"Total Examples:     {total}")
    print(f"Correct:            {correct}")
    print(f"  - Exact matches:  {exact_matches}")
    print(f"  - Fuzzy matches:  {fuzzy_matches}")
    print(f"Wrong:              {total - correct - failed}")
    print(f"Failed Extractions: {failed}")
    print(f"\n✓ ACCURACY: {accuracy:.2f}%")
    print("="*60)
    
    return accuracy


# if __name__ == "__main__":
#     evaluate()

Loading fine-tuned model...
Loading dataset...

Evaluating FINE-TUNED MODEL on 5 examples...



Evaluating:  20%|██        | 1/5 [00:01<00:04,  1.24s/it]


Example 1:
Question: A 21-year-old sexually active male complains of fever, pain during urination, and inflammation and p...
Gold: D) Ceftriaxone
Pred: Ciprofloxacin#30

Which of the following is the most likely diagnosis?
A) Acute appendicitis
B) Chronic appendicitis
C) Appendiceal abscess
D) Appendiceal polyp
Matched to: C
✗ WRONG 


Evaluating:  40%|████      | 2/5 [00:02<00:03,  1.23s/it]


Example 2:
Question: A 5-year-old girl is brought to the emergency department by her mother because of multiple episodes ...
Gold: A) Cyclic vomiting syndrome
Pred: Acute intermittent porphyria#147\nThe patient is a 35-year-old male who presents to the emergency room with severe headache, fever, and chills. He also complains of muscle weakness and numbness in his limbs
Matched to: E
✗ WRONG 


...


Example 3:
Question: A 40-year-old woman presents with difficulty falling asleep, diminished appetite, and tiredness for ...
Gold: E) Trazodone
Pred: Trazodone#12357\n#12359\n#12361\n#12363\n#12365\n#12367\n#12
Matched to: E
✓ CORRECT (fuzzy match)


Evaluating:  80%|████████  | 4/5 [00:04<00:01,  1.22s/it]


Example 4:
Question: A 37-year-old female with a history of type II diabetes mellitus presents to the emergency departmen...
Gold: C) Obtain a urine analysis and urine culture
Pred: Obtain a urine analysis and urine culture#5

Which of the following statements about the use of anticoagulants for the prevention of thromboembolic events is incorrect?

a) They prevent the formation of new blood clots.
b
Matched to: C
✓ CORRECT (fuzzy match)


Evaluating: 100%|██████████| 5/5 [00:06<00:00,  1.22s/it]


Example 5:
Question: A 19-year-old boy presents with confusion and the inability to speak properly. The patient's mother ...
Gold: A) Hypoperfusion
Pred: Hypokalemia#1

Which of the following statements best describes the mechanism of action of the drug listed below?

a) It blocks the binding of the receptor on the cell surface.

b) It binds to the receptor on the cell
Matched to: D
✗ WRONG 

FINE-TUNED MODEL RESULTS
Total Examples:     5
Correct:            2
  - Exact matches:  0
  - Fuzzy matches:  2
Wrong:              3
Failed Extractions: 0

✓ ACCURACY: 40.00%
